# 04 — Treino e avaliação da WiSARD

**Objetivo**: treinar a WiSARD em dois esquemas de binarização (Otsu e Thermometer-HSV) e avaliar a performance como classificador binário de patches (cacho/não-cacho).

**Esta é a primeira medição real do projeto.** Os notebooks anteriores prepararam o terreno (dataset → patches → vetores binários). Aqui finalmente treinamos a WiSARD e calculamos precisão, recall e F1 — métricas comparáveis às do trabalho do Sasse et al.

**Decisões fixadas:**
- **Modelo**: WiSARD vanilla (não ClusWiSARD nesta primeira rodada).
- **Esquemas**: Otsu e Thermometer-HSV.
- **`addressSize`**: 12 (compatível com ambos os tamanhos de vetor).
- **Conjunto de avaliação**: valid (88 imagens). Test fica reservado para a medição final.
- **Resultados em CSV** para uso em comparações posteriores.

**API do wisardpkg**: a versão atual exige um objeto `wp.DataSet` populado linha a linha — `model.train(lista, rotulos)` não funciona. As células abaixo já usam o formato correto.

**Importante sobre o significado**: estas métricas são de **classificação por patch**, não de detecção de cachos. Um patch positivo classificado corretamente significa "este pedaço da imagem tem cacho". A contagem de cachos virá depois (notebook 05), via agregação por componentes conexas.

## 0. Setup

In [1]:
from pathlib import Path
import sys
import time
from datetime import datetime

import numpy as np
import pandas as pd
import wisardpkg as wp

# >>> AJUSTE OS CAMINHOS <<<
BIN_DIR = Path("./binarized")
RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)

PATCH_SIZE = 32
SCHEMES_TO_TEST = ["thermometer_hsv"]
ADDRESS_SIZE = 32

print(f"Esquemas a rodar: {SCHEMES_TO_TEST}")
print(f"addressSize: {ADDRESS_SIZE}")
print(f"BIN_DIR: {BIN_DIR.resolve()}")
print(f"RESULTS_DIR: {RESULTS_DIR.resolve()}")

Esquemas a rodar: ['otsu', 'thermometer_hsv']
addressSize: 12
BIN_DIR: C:\Users\thaty\Documents\ASO\Redes Neurais sem Peso\binarized
RESULTS_DIR: C:\Users\thaty\Documents\ASO\Redes Neurais sem Peso\results


## 1. Utilitários de log

Mensagens com timestamp e `flush=True` para ver progresso em tempo real, especialmente útil para operações que demoram (populamento de DataSet, treino do thermometer).

In [2]:
def log(msg):
    """Imprime com timestamp e força flush."""
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {msg}", flush=True)


def log_mem():
    """Imprime uso de memória do processo (se psutil estiver disponível)."""
    try:
        import psutil
        process = psutil.Process()
        mem_mb = process.memory_info().rss / 1e6
        log(f"  memória do processo: {mem_mb:.0f} MB")
    except ImportError:
        pass

## 2. Carregar binarizações

In [3]:
def load_binarized(scheme_name, split, patch_size=PATCH_SIZE, bin_dir=BIN_DIR):
    """Carrega dataset binarizado e desempacota bits."""
    path = bin_dir / f"bin_{scheme_name}_{split}_{patch_size}.npz"
    data = np.load(path)
    X_packed = data["X_packed"]
    y = data["y"]
    bits_per_patch = int(data["bits_per_patch"])
    X = np.unpackbits(X_packed, axis=1)[:, :bits_per_patch]
    return X, y


# Sanidade: verifica que todos os arquivos esperados existem
for scheme in SCHEMES_TO_TEST:
    for split in ["train", "valid"]:
        path = BIN_DIR / f"bin_{scheme}_{split}_{PATCH_SIZE}.npz"
        status = "OK " if path.exists() else "FALTANDO"
        print(f"  [{status}] {path}")

  [FALTANDO] binarized\bin_otsu_train_24.npz
  [FALTANDO] binarized\bin_otsu_valid_24.npz
  [OK ] binarized\bin_thermometer_hsv_train_24.npz
  [OK ] binarized\bin_thermometer_hsv_valid_24.npz


## 3. Métricas

Cálculo manual de TP/FP/FN/TN e derivadas (precisão, recall, F1, acurácia). Inclui teste de unidade com caso conhecido.

In [4]:
def compute_metrics(y_true, y_pred, positive_label="pos"):
    """Calcula TP, FP, FN, TN, precisão, recall, F1 e acurácia."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    is_pos_true = (y_true == positive_label)
    is_pos_pred = (y_pred == positive_label)
    
    tp = int(np.sum( is_pos_true &  is_pos_pred))
    fp = int(np.sum(~is_pos_true &  is_pos_pred))
    fn = int(np.sum( is_pos_true & ~is_pos_pred))
    tn = int(np.sum(~is_pos_true & ~is_pos_pred))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    
    return {
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
    }


# Teste com caso conhecido: TP=2, FN=1, FP=1, TN=1
y_true_test = np.array(["pos", "pos", "pos", "neg", "neg"])
y_pred_test = np.array(["pos", "neg", "pos", "pos", "neg"])
m = compute_metrics(y_true_test, y_pred_test)
assert m["tp"] == 2 and m["fp"] == 1 and m["fn"] == 1 and m["tn"] == 1
assert abs(m["precision"] - 2/3) < 1e-6
assert abs(m["recall"] - 2/3) < 1e-6
print("Teste de métricas: OK")
print("Caso de teste:", m)

Teste de métricas: OK
Caso de teste: {'tp': 2, 'fp': 1, 'fn': 1, 'tn': 1, 'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'f1': 0.6666666666666666, 'accuracy': 0.6}


## 4. Função principal: treino e avaliação de um esquema

Esta é a função que faz o trabalho pesado para cada esquema:
1. Carrega train e valid binarizados.
2. Popula um `wp.DataSet` linha a linha (sem materializar lista de listas).
3. Treina a WiSARD.
4. Classifica o conjunto de validação.
5. Calcula métricas e mede tempos.

In [5]:
def run_experiment(scheme):
    log(f"=== Iniciando esquema: {scheme} ===")

    # ---- Carregar dados ----
    log("Carregando train (pode demorar para thermometer)...")
    t0 = time.time()
    X_train_bin, y_train = load_binarized(scheme, "train")
    log(f"  train carregado: {X_train_bin.shape} em {time.time()-t0:.1f}s")
    log_mem()

    log("Carregando valid...")
    X_valid_bin, y_valid = load_binarized(scheme, "valid")
    log(f"  valid carregado: {X_valid_bin.shape}")
    log_mem()

    # ---- Verificar divisibilidade ----
    bits = X_train_bin.shape[1]
    if bits % ADDRESS_SIZE != 0:
        raise ValueError(
            f"bits_per_patch ({bits}) não é múltiplo de addressSize ({ADDRESS_SIZE}). "
            f"Resto: {bits % ADDRESS_SIZE}"
        )
    log(f"  bits/patch = {bits}, addressSize = {ADDRESS_SIZE}, "
        f"n_RAMs/discriminador = {bits // ADDRESS_SIZE}")

    # ---- Popular wp.DataSet de treino ----
    # API atual exige DataSet; populamos linha a linha direto do array NumPy
    # para evitar materializar uma lista de listas gigante.
    log("Populando wp.DataSet de treino...")
    t0 = time.time()
    train_ds = wp.DataSet()
    n_train = len(X_train_bin)
    for i in range(n_train):
        train_ds.add(X_train_bin[i].tolist(), str(y_train[i]))
        if (i + 1) % 20000 == 0:
            log(f"  {i+1}/{n_train} amostras adicionadas")
    log(f"  DataSet train montado: {train_ds.size()} amostras em {time.time()-t0:.1f}s")
    log_mem()

    del X_train_bin

    # ---- Popular wp.DataSet de validação ----
    log("Populando wp.DataSet de validação...")
    t0 = time.time()
    valid_ds = wp.DataSet()
    n_valid = len(X_valid_bin)
    for i in range(n_valid):
        valid_ds.add(X_valid_bin[i].tolist())  # sem rótulo: só para classificar
    log(f"  DataSet valid montado: {valid_ds.size()} amostras em {time.time()-t0:.1f}s")
    log_mem()

    y_valid_list = y_valid.tolist()
    del X_valid_bin

    # ---- Treino ----
    log(f"Criando modelo Wisard(addressSize={ADDRESS_SIZE})...")
    model = wp.Wisard(ADDRESS_SIZE)

    log(f"Treinando com {train_ds.size()} amostras...")
    t0 = time.time()
    model.train(train_ds)
    train_time = time.time() - t0
    log(f"  TREINO CONCLUÍDO em {train_time:.2f}s")
    log_mem()

    del train_ds

    # ---- Inferência ----
    log(f"Classificando {valid_ds.size()} amostras de validação...")
    t0 = time.time()
    y_pred = model.classify(valid_ds)
    inference_time = time.time() - t0
    inf_per_sample_ms = (inference_time / valid_ds.size()) * 1000
    log(f"  inferência concluída em {inference_time:.2f}s "
        f"({inf_per_sample_ms:.3f} ms por amostra)")

    # ---- Métricas ----
    metrics = compute_metrics(y_valid_list, y_pred)

    log(f"  RESULTADOS - {scheme}:")
    log(f"    F1-score:  {metrics['f1']:.4f}")
    log(f"    Precisão:  {metrics['precision']:.4f}")
    log(f"    Recall:    {metrics['recall']:.4f}")
    log(f"    Acurácia:  {metrics['accuracy']:.4f}")
    log(f"    TP={metrics['tp']}  FP={metrics['fp']}  FN={metrics['fn']}  TN={metrics['tn']}")

    return {
        "timestamp":               datetime.now().isoformat(timespec="seconds"),
        "scheme":                  scheme,
        "model":                   "Wisard",
        "address_size":            ADDRESS_SIZE,
        "patch_size":              PATCH_SIZE,
        "n_train":                 n_train,
        "n_valid":                 len(y_valid_list),
        "bits_per_patch":          bits,
        "train_time_s":            round(train_time, 3),
        "inference_time_s":        round(inference_time, 3),
        "inference_per_sample_ms": round(inf_per_sample_ms, 4),
        **{k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()},
    }

## 5. Rodar os experimentos

Aqui é onde o trabalho acontece. Cada esquema é tratado isoladamente; se um falhar, os demais continuam.

In [6]:
results = []
for scheme in SCHEMES_TO_TEST:
    try:
        result = run_experiment(scheme)
        results.append(result)
    except Exception as e:
        log(f"ERRO em {scheme}: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        log(f"  pulando {scheme}, continuando com o próximo")

log("")
log(f"Concluído. {len(results)}/{len(SCHEMES_TO_TEST)} esquemas com sucesso.")

[18:33:10] === Iniciando esquema: otsu ===
[18:33:10] Carregando train (pode demorar para thermometer)...
[18:33:10] ERRO em otsu: FileNotFoundError: [Errno 2] No such file or directory: 'binarized\\bin_otsu_train_24.npz'
[18:33:10]   pulando otsu, continuando com o próximo
[18:33:10] === Iniciando esquema: thermometer_hsv ===
[18:33:10] Carregando train (pode demorar para thermometer)...


Traceback (most recent call last):
  File "C:\Users\thaty\AppData\Local\Temp\ipykernel_7492\2585246250.py", line 4, in <module>
    result = run_experiment(scheme)
  File "C:\Users\thaty\AppData\Local\Temp\ipykernel_7492\3345378194.py", line 7, in run_experiment
    X_train_bin, y_train = load_binarized(scheme, "train")
                           ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "C:\Users\thaty\AppData\Local\Temp\ipykernel_7492\3974001370.py", line 4, in load_binarized
    data = np.load(path)
  File "C:\Users\thaty\anaconda3\Lib\site-packages\numpy\lib\_npyio_impl.py", line 454, in load
    fid = stack.enter_context(open(os.fspath(file), "rb"))
                              ~~~~^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'binarized\\bin_otsu_train_24.npz'


[18:33:12]   train carregado: (296685, 6912) em 2.4s
[18:33:12]   memória do processo: 2197 MB
[18:33:12] Carregando valid...
[18:33:15]   valid carregado: (226382, 6912)
[18:33:15]   memória do processo: 3765 MB
[18:33:15]   bits/patch = 6912, addressSize = 12, n_RAMs/discriminador = 576
[18:33:15] Populando wp.DataSet de treino...
[18:33:23]   20000/296685 amostras adicionadas
[18:33:29]   40000/296685 amostras adicionadas
[18:33:34]   60000/296685 amostras adicionadas
[18:33:39]   80000/296685 amostras adicionadas
[18:33:44]   100000/296685 amostras adicionadas
[18:33:48]   120000/296685 amostras adicionadas
[18:33:53]   140000/296685 amostras adicionadas
[18:33:58]   160000/296685 amostras adicionadas
[18:34:03]   180000/296685 amostras adicionadas
[18:34:08]   200000/296685 amostras adicionadas
[18:34:13]   220000/296685 amostras adicionadas
[18:34:17]   240000/296685 amostras adicionadas
[18:34:23]   260000/296685 amostras adicionadas
[18:34:27]   280000/296685 amostras adicionad

## 6. Tabela comparativa

In [7]:
results_df = pd.DataFrame(results)

display_cols = ["scheme", "f1", "precision", "recall", "accuracy", 
                "tp", "fp", "fn", "tn",
                "train_time_s", "inference_per_sample_ms"]
results_df[display_cols]

,scheme,f1,precision,recall,accuracy,tp,fp,fn,tn,train_time_s,inference_per_sample_ms
0,thermometer_hsv,0.1532,0.0841,0.8563,0.7069,6000,65342,1007,154033,51.238,0.3979


## 7. Salvar resultados em CSV

Modo append: rodar este notebook múltiplas vezes (com outros hiperparâmetros, outros esquemas) acumula tudo no mesmo CSV.

In [8]:
csv_path = RESULTS_DIR / "experiments.csv"

if csv_path.exists():
    existing = pd.read_csv(csv_path)
    combined = pd.concat([existing, results_df], ignore_index=True)
    print(f"Arquivo já existia com {len(existing)} linhas. Adicionando {len(results_df)}.")
else:
    combined = results_df
    print(f"Criando arquivo novo com {len(results_df)} linhas.")

combined.to_csv(csv_path, index=False)
print(f"Salvo em: {csv_path}")
print(f"Total de experimentos no CSV: {len(combined)}")

Arquivo já existia com 5 linhas. Adicionando 1.
Salvo em: results\experiments.csv
Total de experimentos no CSV: 6


## 8. Diagnóstico: distribuição de erros

Olhar só F1 não diz tudo. Vale entender que tipo de erro a WiSARD está cometendo: ela está **errando para mais** (muitos falsos positivos, baixa precisão) ou **para menos** (muitos falsos negativos, baixo recall)? Cada tipo de erro afeta a contagem final de forma diferente.

In [9]:
for r in results:
    print(f"\n--- {r['scheme']} ---")
    total_pos = r['tp'] + r['fn']
    total_neg = r['fp'] + r['tn']
    print(f"  Cachos detectados: {r['tp']:>6} de {total_pos} ({r['recall']*100:.1f}%)")
    print(f"  Cachos perdidos:   {r['fn']:>6} de {total_pos} ({(1-r['recall'])*100:.1f}%)")
    print(f"  Falsos alarmes:    {r['fp']:>6} de {total_neg} ({r['fp']/max(total_neg,1)*100:.2f}%)")
    ratio = r['fp'] / max(r['tp'], 1)
    print(f"  Razão FP/TP:       {ratio:.2f}  (alarmes para cada cacho detectado)")
    if ratio > 5:
        print("    ATENCAO: Muitos falsos alarmes — agregação por componentes conexas vai sofrer")
    elif ratio < 1:
        print("    OK: Razão saudável")


--- thermometer_hsv ---
  Cachos detectados:   6000 de 7007 (85.6%)
  Cachos perdidos:     1007 de 7007 (14.4%)
  Falsos alarmes:     65342 de 219375 (29.79%)
  Razão FP/TP:       10.89  (alarmes para cada cacho detectado)
    ATENCAO: Muitos falsos alarmes — agregação por componentes conexas vai sofrer


## 9. Próximos passos

Com o classificador funcionando, os caminhos possíveis são:

**Se F1 ≥ 50%**: ir direto para o notebook 05 (inferência por janela deslizante na imagem inteira + contagem por componentes conexas). É a primeira métrica comparável diretamente com o trabalho do Sasse.

**Se F1 entre 30% e 50%**: vale tentar ClusWiSARD antes — ele pode capturar variabilidade intraclasse melhor que o WiSARD vanilla. Ou testar outros valores de `addressSize`.

**Se F1 < 30%**: revisitar a binarização ou a rotulagem dos patches. Algum esquema novo pode ser necessário.

**Lembre**: comparação direta com YOLO (F1 = 69,9%) não é completamente justa — ele faz detecção de bbox, nós fazemos classificação por patch. Mas é a referência mais próxima possível dada a diferença de abordagem.